# 98 Veeva Link dictionary report (Phase 4, O3)

Task O3 of `95_veeva_link_phase4_working_plan.md`. Read-only on the frozen snapshot and the config tables; proposes changes, never edits `focus_dictionary_v1` / `keyword_list_v2` (P4-D7).

* (0) `config/focus_names_seen_v1`: every distinct focus name per score table in the snapshot, in scope or not (written once, append-only).
* (a) focus names in the current latest rows not in the seen list; (b) `focus_dictionary_v1` names no longer present.
* (c) titles of trials, events and publications where an include keyword and an exclude keyword of the same tag both hit; near misses: titles with in-scope disease words but no include hit.
* (d) review sheet for X5 (Medical Affairs / brand).

Checks: N8 0 new names vs the seen list on the current snapshot and every v1 name still present; N9 every report row has counts and up to 5 sample titles. Output: `OUT/ops/dictionary_report/run=<ts>/`.

In [0]:
from pyspark.sql import functions as F
import pandas as pd, datetime as dt

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
SNAP = OUT + "snapshot/load=2025-06-30/"
RUN_TS = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")
P = f"{OUT}ops/dictionary_report/run={RUN_TS}/"
SEEN = OUT + "config/focus_names_seen_v1"
SCORE = {"clinical_leader": ["FOCUS_NAME", "TYPE"], "focus_area": ["FOCUS_NAME"], "digital_reach_focus_area": ["FOCUS_NAME"]}

def names(t):   # current latest rows (snapshot; replace with the new snapshot folder after a refresh)
    k = SCORE[t]
    return (spark.read.parquet(SNAP + t).groupBy(*k).agg(F.countDistinct("SE_ID").alias("experts"))
                 .withColumn("source_table", F.lit(t)).withColumn("cl_type", F.col("TYPE") if "TYPE" in k else F.lit(None).cast("string"))
                 .select("source_table", F.col("FOCUS_NAME").alias("focus_name"), "cl_type", "experts"))
cur = names("clinical_leader").unionByName(names("focus_area")).unionByName(names("digital_reach_focus_area")).toPandas()

try:
    seen = spark.read.parquet(SEEN).toPandas()
    seen_written = False
except Exception:   # first run: write the seen list once (P4-D2, append-only)
    seen = cur.assign(first_seen_snapshot="load=2025-06-30", seen_version="v1")
    spark.createDataFrame(seen).write.parquet(SEEN)   # default save mode fails if the folder exists
    seen_written = True

key = ["source_table", "focus_name", "cl_type"]
m = cur.merge(seen[key].assign(_s=1), on=key, how="left")
new_names = m[m._s.isna()].drop(columns="_s").sort_values("experts", ascending=False)
fd = spark.read.parquet(OUT + "config/focus_dictionary_v1").toPandas()
fd_key = fd[key].drop_duplicates()
r = fd_key.merge(cur[key].assign(_c=1), on=key, how="left")
removed = r[r._c.isna()].drop(columns="_c")
print(f"seen list rows {len(seen):,} (written this run: {seen_written}) | current names {len(cur):,} | new vs seen {len(new_names)} | v1 names {len(fd_key)} | v1 names no longer present {len(removed)}")
display(cur.groupby("source_table").agg(names=("focus_name", "size"), experts_max=("experts", "max")).reset_index())

In [0]:
kw = spark.read.parquet(OUT + "config/keyword_list_v2").toPandas()
TXT = {"clinical_trial": "NAME", "medical_event": "NAME", "publication": "TITLE"}
DISEASE = ["%kidney%", "%renal%", "%nephr%", "%heart failure%", "%cardiomyopath%", "%diabet%", "%albuminuria%", "%proteinuria%",
           "%mineralocorticoid%", "%aldosterone%", "%cardiorenal%"]

def any_like(col, pats):
    e = F.lit(False)
    for p in pats:
        e = e | F.lower(F.col(col)).like(p.lower())
    return e

rows, samples = [], []
for t, c in TXT.items():
    d = spark.read.parquet(SNAP + t).select(F.col(c).alias("txt")).filter(F.col("txt").isNotNull()).dropDuplicates(["txt"])
    inc_all = kw[kw.rule_type == "include"].like_pattern.tolist()
    for tag in sorted(kw[kw.rule_type == "exclude"].tag.unique()):   # include and exclude of the same tag both hit
        inc = kw[(kw.tag == tag) & (kw.rule_type == "include")].like_pattern.tolist()
        exc = kw[(kw.tag == tag) & (kw.rule_type == "exclude")].like_pattern.tolist()
        hit = d.filter(any_like("txt", inc) & any_like("txt", exc))
        rows.append({"part": "c1 include+exclude", "source": t, "item": tag, "n_titles": hit.count(),
                     "sample_titles": " || ".join(r.txt[:120] for r in hit.limit(5).collect())})
    for w in DISEASE:   # disease word, no include keyword of any tag
        hit = d.filter(F.lower("txt").like(w) & ~any_like("txt", inc_all))
        rows.append({"part": "c2 near miss", "source": t, "item": w, "n_titles": hit.count(),
                     "sample_titles": " || ".join(r.txt[:120] for r in hit.limit(5).collect())})
rep_c = pd.DataFrame(rows)
display(rep_c.pivot_table(index=["part", "item"], columns="source", values="n_titles", aggfunc="sum").reset_index())

In [0]:
sheet = pd.concat([
    new_names.assign(part="a new focus name", item=new_names.focus_name, n_titles=None, sample_titles=None, source=new_names.source_table),
    removed.assign(part="b v1 name not present", item=removed.focus_name, n_titles=None, sample_titles=None, source=removed.source_table, experts=0),
    rep_c[rep_c.n_titles > 0]], ignore_index=True)[["part", "source", "item", "cl_type", "experts", "n_titles", "sample_titles"]]
sheet["proposed_action"] = ""; sheet["reviewer_decision"] = ""; sheet["reviewer"] = ""
from pyspark.sql.types import StructType, StructField, StringType
def to_spark(x):   # all columns as strings with an explicit schema (all-null columns would be VOID)
    x = x.astype(object).where(x.notna(), None).astype(object)
    x = x.apply(lambda s: s.map(lambda v: None if v is None else str(v)))
    return spark.createDataFrame(x, StructType([StructField(c, StringType(), True) for c in x.columns]))
for name, df in {"new_names": new_names, "removed_names": removed, "keyword_report": rep_c, "review_sheet": sheet}.items():
    x = df.copy()
    if len(x):
        to_spark(x).write.mode("overwrite").parquet(P + name)
to_spark(sheet).coalesce(1).write.mode("overwrite").option("header", True).csv(P + "review_sheet_csv")
rc = rep_c[rep_c.n_titles > 0]
chk = pd.DataFrame([
    {"check": "N8", "name": "0 new names vs seen list; all v1 names present", "value": f"new {len(new_names)} | v1 missing {len(removed)}", "pass": len(new_names) == 0 and len(removed) == 0},
    {"check": "N9", "name": "report rows with hits have counts and samples", "value": f"{len(rc)} rows with hits",
     "pass": bool(rc.n_titles.notna().all() and (rc.sample_titles.str.len() > 0).all())}])
chk["run_ts"] = RUN_TS
spark.createDataFrame(chk.astype(str)).write.mode("overwrite").parquet(P + "checks")
display(chk)
print("review sheet rows:", len(sheet), "| written:", P)